# 2025/3/9

In [1]:
import os
import matplotlib.pyplot as plt
import random
import pandas as pd
import json
import tqdm

In [ ]:
from PIL import Image
from transformers import pipeline

classifier = pipeline("image-classification", model="Falconsai/nsfw_image_detection")
# img = Image.open("daiyousei.png")
# print(classifier(img))

In [2]:
dataset_path = '../data/touhou'
datasets = os.listdir(dataset_path)
print(f"角色类别数：{len(datasets)}")

角色类别数：140


In [3]:
# 统计每个角色类别的图片数量
dataset_counts = {}
for dataset in datasets:
    path = os.path.join(dataset_path, dataset)
    if os.path.isdir(path):
        num_images = len(os.listdir(path)) // 2     # //2是因为有标签文件
        dataset_counts[dataset] = num_images
# 将结果转换为DataFrame
df = pd.DataFrame(list(dataset_counts.items()), columns=['角色', '图片数量'])
# 按图片数量降序排序
df = df.sort_values(by='图片数量', ascending=False)
# 显示后20个角色和其图片数量
print(df.tail(20))

                        角色  图片数量
60                konngara    81
123         tsukumo_benben    78
62                kotohime    74
77                   meira    68
3           asakura_rikako    60
74           maribel_hearn    58
129  watatsuki_no_yorihime    56
105                   sara    51
93                  orange    44
33       iizunamaru_megumu    42
106                 sariel    42
99                    rika    41
65           kurokoma_saki    40
27          horikawa_raiko    39
102                ruukoto    35
78       merlin_prismriver    31
24        himemushi_momoyo    29
121         toutetsu_yuuma    19
136        yatadera_narumi    10
107            satsuki_rin     9


In [4]:
# # 转换英文角色名为中文角色名
# cvt_name = {}
# try:
#     for chara, count in dataset_counts.items():
#         print(chara, count)
#         name_cn = input(f"请输入{chara}的中文角色名：").strip()
#         # dataset_counts[name_cn] = dataset_counts.pop(chara)
#         cvt_name[chara] = name_cn
# finally:
#     json.dump(cvt_name, open('cvt_name.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=4)
cvt_name = json.load(open('cvt_name.json', 'r', encoding='utf-8'))

In [ ]:
# 分别按中文名和英文名保存统计结果
df.to_csv('dataset_counts.csv', index=False, encoding='utf-8')

In [ ]:
df_cn = df.copy()
df_cn['角色'] = df_cn['角色'].map(cvt_name)
df_cn.to_csv('dataset_counts_cn.csv', index=False, encoding='utf-8')

In [ ]:
with open('characters.txt', 'r', encoding='utf-8') as f:
    characters = f.readlines()
characters = [char.strip() for char in characters]
existing_characters = set(df_cn['角色'].tolist())
missing_characters = [char for char in characters if char not in existing_characters]
with open('missing_characters.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(missing_characters))


In [ ]:
# 随机抽取10个图片数大于100的角色，随机抽取50张图，检测每张图片是否属于nsfw，并统计各个角色的结果
# 随机抽取10个角色
random_characters = random.sample(list(df[df['图片数量'] > 200]['角色']), 15)
nsfw_counts = {}
for char in random_characters:
    char_path = os.path.join(dataset_path, char)
    images = os.listdir(char_path)
    images = [os.path.join(char_path, img) for img in images if img.endswith('.png')]
    images = random.sample(images, 60)
    nsfw_count = {'nsfw': 0, 'safe': 0}
    for img_path in images:
        img = Image.open(img_path)
        result = classifier(img)
        if result[0]['label'] == 'nsfw':
            nsfw_count['nsfw'] += 1
        else:
            nsfw_count['safe'] += 1
    nsfw_count['ratio'] = nsfw_count['nsfw'] / (nsfw_count['nsfw'] + nsfw_count['safe'])
    print(f"{char}：{nsfw_count}")
    nsfw_counts[char] = nsfw_count
print(f'average ratio: {sum([nsfw_counts[char]["ratio"] for char in nsfw_counts]) / len(nsfw_counts)}')


In [ ]:
# 随机抽取10个图片数小于200的角色，检测每张图片是否属于nsfw，并统计各个角色的结果
random_characters = random.sample(list(df[df['图片数量'] < 200]['角色']), 10)
nsfw_counts_2 = {}
for char in random_characters:
    char_path = os.path.join(dataset_path, char)
    images = os.listdir(char_path)
    images = [os.path.join(char_path, img) for img in images if img.endswith('.png')]
    # images = random.sample(images, 50)
    nsfw_count = {'nsfw': 0, 'safe': 0}
    for img_path in images:
        img = Image.open(img_path)
        result = classifier(img)
        if result[0]['label'] == 'nsfw':
            nsfw_count['nsfw'] += 1
        else:
            nsfw_count['safe'] += 1
    nsfw_count['ratio'] = nsfw_count['nsfw'] / (nsfw_count['nsfw'] + nsfw_count['safe'])
    print(f"{char}：{nsfw_count}")
    nsfw_counts_2[char] = nsfw_count
print(f'average ratio: {sum([nsfw_counts_2[char]["ratio"] for char in nsfw_counts_2]) / len(nsfw_counts_2)}')

# 2025/3/10

In [ ]:
# 统计所有类别角色的nsfw图片数量和比例
r18_counts = {}
r18_records = {}
chars = df['角色'].tolist()
chars = tqdm.tqdm(chars)
try:
    for char in chars:
        chars.set_description(f"Processing {char}")
        char_path = os.path.join(dataset_path, char)
        images = os.listdir(char_path)
        images = [os.path.join(char_path, img) for img in images if img.endswith(('.png', '.jpg', '.jpeg'))]
        # images = tqdm.tqdm(images)

        nsfw_count = {'nsfw': 0, 'safe': 0, 'name': cvt_name[char]}
        record = []
        for img_path in images:
            img = Image.open(img_path)
            result = classifier(img)
            if result[0]['label'] == 'nsfw':
                nsfw_count['nsfw'] += 1
                record.append(img_path)
            else:
                nsfw_count['safe'] += 1
            chars.set_postfix(nsfw=nsfw_count['nsfw'], safe=nsfw_count['safe'])

        nsfw_count['ratio'] = nsfw_count['nsfw'] / (nsfw_count['nsfw'] + nsfw_count['safe'])
        r18_counts[char] = nsfw_count
        r18_records[char] = record
        tqdm.tqdm.write(f"{char}：{nsfw_count}")
    print(f'average ratio: {sum([r18_counts[char]["ratio"] for char in r18_counts]) / len(r18_counts)}')
finally:
    with open('r18_counts.json', 'w', encoding='utf-8') as f:
        json.dump(r18_counts, f, ensure_ascii=False, indent=4)
    with open('r18_records.json', 'w', encoding='utf-8') as f:
        json.dump(r18_records, f, ensure_ascii=False, indent=4)


In [ ]:
# 排序找出nsfw比例最高的角色
sorted_r18_counts = sorted(r18_counts.items(), key=lambda x: x[1]['ratio'], reverse=True)
for i in range(10):
    print(sorted_r18_counts[i])

In [ ]:
# 随机抽取10个角色各5张nsfw图片显示
import random
import matplotlib.pyplot as plt

# samples = random.sample(r18_counts.keys(), 10)
fig, axs = plt.subplots(10, 5, figsize=(20, 20))
for i, (char, count) in enumerate(sorted_r18_counts[:10]):
    records = r18_records[char]
    nsfw_images = random.sample(records, 5)
    for j, img_path in enumerate(nsfw_images):
        img = Image.open(img_path)
        axs[i, j].imshow(img)
        axs[i, j].axis('off')
    plt.tight_layout()
plt.show()


In [ ]:
# 将所有nsfw图片按角色移动到一个文件夹，并删除同名meta.json文件
import shutil

dataset_path = '../data/touhou'
r18_records = json.load(open('r18_records.json', 'r', encoding='utf-8'))
for char, records in r18_records.items():
    char_path = os.path.join('../data/nsfw_dataset', char)
    os.makedirs(char_path, exist_ok=True)
    cnt = 0
    for img_path in records:
        if not os.path.exists(img_path):
            continue
        shutil.move(img_path, char_path)
        # filename = os.path.basename(img_path).split('.')[0]
        # json_file_path = f'.{filename}_meta.json'
        # os.remove(os.path.join(dataset_path, char, json_file_path))
        cnt += 1
    print(f'{char}: {cnt} images moved')





In [ ]:
# 重新统计每个角色的图片数量
new_datasets_count = {}
for dataset in datasets:
    path = os.path.join(dataset_path, dataset)
    images = os.listdir(path)
    images = [os.path.join(path, img) for img in images if img.endswith(('.png', '.jpg', '.jpeg'))]
    new_datasets_count[dataset] = len(images)
    print(f'{dataset}: {len(images)} images')

# 转换为df并排序后保存
new_df = pd.DataFrame(new_datasets_count.items(), columns=['角色', '图片数量'])
new_df = new_df.sort_values(by='图片数量', ascending=False)
new_df.to_csv('new_datasets_count.csv', index=False)
# 转换为中文并保存
new_df_cn = new_df.copy()
new_df_cn['角色'] = new_df_cn['角色'].map(cvt_name)
new_df_cn.to_csv('new_datasets_count_cn.csv', index=False)

In [ ]:
# 输出new_df_cn的统计信息
print(new_df_cn.describe())

In [ ]:
result = classifier(Image.open('mouko2.png'))
print(result)

In [ ]:
from transformers import pipeline

model = pipeline("image-classification", model="LukeJacob2023/nsfw-image-detector")

In [ ]:
print(model(Image.open('aki.png')))

In [ ]:
# 重新筛选nsfw图片，对于非nsfw的图片重新返回到原文件夹
import os
import shutil

nsfw_dataset_path = '../data/nsfw_dataset'
to_be_saves = {}
for char in os.listdir(nsfw_dataset_path):
    print(f'Processing {char}...')
    char_path = os.path.join(nsfw_dataset_path, char)
    images = os.listdir(char_path)
    to_be_saves[char] = []
    for img in images:
        img_path = os.path.join(char_path, img)
        result = model(Image.open(img_path))
        if result[0]['label'] != 'drawings':
            continue
            # shutil.move(img_path, os.path.join(dataset_path, char, img))
            # to_be_saves[char].append(img_path)
        elif result[1]['label'] in ['hentai', 'porn', 'sexy'] and result[1]['score'] > 0.01:
            continue
        to_be_saves[char].append(img_path)
    print(f'total {len(images)} images, {len(to_be_saves[char])} images to be saved')
json.dump(to_be_saves, open('to_be_saves.json', 'w'), indent=4)

In [ ]:
# 根据to_be_saves.json将图片重新返回到原文件夹
import json

to_be_saves = json.load(open('to_be_saves.json', 'r'))
for char, images in to_be_saves.items():
    print(f'Processing {char}...')
    for img in images:
        shutil.move(img, os.path.join(dataset_path, char, os.path.basename(img)))

In [ ]:
# 重新统计每个角色的图片数量
new_datasets_count = {}
for dataset in datasets:
    path = os.path.join(dataset_path, dataset)
    images = os.listdir(path)
    images = [os.path.join(path, img) for img in images if img.endswith(('.png', '.jpg', '.jpeg'))]
    new_datasets_count[dataset] = len(images)
    print(f'{dataset}: {len(images)} images')

# 转换为df并排序后保存
new_df = pd.DataFrame(new_datasets_count.items(), columns=['角色', '图片数量'])
new_df = new_df.sort_values(by='图片数量', ascending=False)
# new_df.to_csv('new_datasets_count.csv', index=False)
# 转换为中文并保存
new_df_cn = new_df.copy()
new_df_cn['角色'] = new_df_cn['角色'].map(cvt_name)
# new_df_cn.to_csv('new_datasets_count_cn.csv', index=False)

In [ ]:
new_df_cn.loc[new_df_cn['角色'] == '博丽灵梦', '图片数量'].values[0]

In [ ]:
# 统计所有角色的图片数量距离目标的差距

with open('characters.json', 'r', encoding='utf-8') as f:
    characters_json = json.load(f)

target = {}
cvt_name_to_en = {v: k for k, v in cvt_name.items()}
for origin, characters in characters_json.items():
    if origin == 'others' or origin.endswith(('01', '02', '03', '04', '05')):
        target_count = 500
    else:
        target_count = 1000
    target[origin] = {}
    for char in characters:
        target[origin][char] = {'目标': target_count}
        target[origin][char]['当前'] = int(new_df_cn.loc[new_df_cn['角色'] == char, '图片数量'].values[0]) if char in cvt_name_to_en else 0
        target[origin][char]['差距'] = target_count - target[origin][char]['当前']


with open('target.json', 'w', encoding='utf-8') as f:
    json.dump(target, f, ensure_ascii=False, indent=4)


In [ ]:
with open('target.json', 'r', encoding='utf-8') as f:
    target = json.load(f)
# print(f'total characters: {sum([len(characters) for characters in target.values()])}')
all_characters = []
for origin, characters in target.items():
    for char in characters:
        all_characters.append(char)
print(f'total characters: {len(all_characters)}')
with open('all_characters.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(all_characters))


total characters: 154
total characters: 154
